In [ ]:
import dsautils.calstatus as cs
from dsautils.dsa_store import DsaStore
from astropy.time import Time
import time
import datetime
import yaml
from dsacalib.weights import average_beamformer_solutions
import glob
import os
import numpy as np
from pkg_resources import resource_filename
import astropy.units as u
from dsautils import cnf
from dsacalib.plotting import summary_plot, plot_current_beamformer_solutions
from dsacalib.plotting import plot_beamformer_weights
from dsacalib.routines import get_files_for_cal, calibrate_measurement_set
from dsacalib.weights import get_good_solution, write_beamformer_solutions
from dsacalib.ms_io import convert_calibrator_pass_to_ms, uvh5_to_ms
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.backends.backend_pdf import PdfPages
import h5py
myconf = cnf.Conf()
ETCD = DsaStore()

## Inspecting beamformer solutions

To plot the beamformer solutions and choose ones to average. First run the cell below, then the following one to plot.

In [ ]:
CORR_PARAMS = myconf.get('corr')
CAL_PARAMS = myconf.get('cal')
MFS_PARAMS = myconf.get('fringe')

REFANTS = CAL_PARAMS['refant']
if isinstance(REFANTS, (str, int)):
    REFANTS = [REFANTS]
MSDIR = CAL_PARAMS['msdir']

BEAMFORMER_DIR = CAL_PARAMS['beamformer_dir']
print(BEAMFORMER_DIR)

ANTENNAS = np.array(list(CORR_PARAMS['antenna_order'].values()))
POLS = CORR_PARAMS['pols_voltage']
ANTENNAS_NOT_IN_BF = CAL_PARAMS['antennas_not_in_bf']
print(ANTENNAS_NOT_IN_BF)
CORR_LIST = list(CORR_PARAMS['ch0'].keys())
CORR_LIST = [int(cl.strip('corr')) for cl in CORR_LIST]

CURRENT_BEAMFORMER_DIR = CAL_PARAMS['bfarchivedir']
print(CURRENT_BEAMFORMER_DIR)
current_weights = ETCD.get_dict('/mon/cal/bfweights')['val']['weight_files']
print(current_weights)

Using the command line, look for the following files: `/operations/beamformer_weights/generated/*.yaml`
Select the latest instance(s) of 1459+716, and enter them into the `bfnames` list below. Be sure to include the date string, e.g., `1459+716_2022-09-06T23:48:09`


In [ ]:
bfnames = [
    '1459+716_2022-09-26T22:29:30'
]
_ = plot_beamformer_weights(
    bfnames,
    ANTENNAS,
    BEAMFORMER_DIR,
    show=True,
    current_weights=current_weights,
    current_weights_dir=BEAMFORMER_DIR
)

## Make the averaged solution and apply

Just run the following cells - no output to inspect.

In [ ]:
with open(
    '{0}/beamformer_weights_{1}.yaml'.format(
        BEAMFORMER_DIR,
        bfnames[0]
    )
) as f:
    latest_solns = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
now = Time(datetime.datetime.utcnow())
now.precision = 0
averaged_files, avg_flags = average_beamformer_solutions(
    bfnames,
    now,
    BEAMFORMER_DIR,
    ANTENNAS,
    58849.0
)

In [ ]:
latest_solns['cal_solutions']['weight_files'] = averaged_files
latest_solns['cal_solutions']['source'] = [
    bfnames[0].split('_')[0]
]
latest_solns['cal_solutions']['caltime'] = [
    float(Time(bfnames[0].split('_')[1]).mjd)
]
for key, value in \
    latest_solns['cal_solutions']['flagged_antennas'].items():
    if 'casa solutions flagged' in value:
        value = value.remove('casa solutions flagged')
# Flag new bad solutions
idxant, idxpol = np.nonzero(avg_flags)
for i, ant in enumerate(idxant):
    key = '{0} {1}'.format(ANTENNAS[ant], POLS[idxpol[i]])
    if key not in \
        latest_solns['cal_solutions']['flagged_antennas'].keys():
        latest_solns['cal_solutions']['flagged_antennas'][key] = []
    latest_solns['cal_solutions']['flagged_antennas'][key] += \
        ['casa solutions flagged']
latest_solns['cal_solutions']['flagged_antennas'] = {
    key: value for key, value in
    latest_solns['cal_solutions']['flagged_antennas'].items()
    if len(value) > 0
}

In [ ]:
with open(
    '{0}/beamformer_weights_{1}.yaml'.format(
        BEAMFORMER_DIR, now.isot
    ),
    'w'
) as file:
    print('writing bf weights')
    _ = yaml.dump(latest_solns, file)

### *Important*

Before running the cell below, start the bfweights_copy service: 
systemctl --user start bfweights_copy.service (anywhere on calibration23)

Then run the cell, and use
journalctl --user-unit bfweights_copy.service -f
to make sure the weights are copied to all corr nodes. 

Finally, be sure to stop the service:
systemctl --user stop bfweights_copy.service

In [ ]:
with open(
    '{0}/beamformer_weights_{1}.yaml'.format(
        BEAMFORMER_DIR,now.isot
    )
) as f:
    latest_solutions = yaml.load(f, Loader=yaml.FullLoader)
ETCD.put_dict(
    '/mon/cal/bfweights',
    {
        'cmd': 'update_weights',
        'val': latest_solns['cal_solutions']
    }
)